# Put a kernel on the FPGA's accelerator, with an LLM — solved

Your board is a **PYNQ-Z1 FPGA** running a RISC-V core (Rocket, 40 MHz) with a small accelerator built in:
**MBP**, four instructions that each work on **eight int8 values at once**. In this notebook:

1. you meet **MBP.MAX8** and do its job by hand,
2. an LLM (DeepSeek, on AWS Bedrock) rewrites a max-pool kernel to use it, **with your FPGA in the loop**,
3. you find out how much of the speedup is the accelerator itself, and
4. you write the kernel yourself and race the LLM.

Run cells with **Shift-Enter**. How it works, in depth: `walkthroughs/` (left, in the file browser).

> **This is the solved copy.** Every cell draws a complete run made on a real board, stored in
> `notebooks/mb_lab/assets/solved_runs.tar.gz`, so it works with no board and no LLM, for when your own run misbehaves,
> or to compare against. The LLM run is `20260925-201916-maxpool2d_s8`, and the "your turn" run is `20260925-202434-maxpool2d_s8`.

In [ ]:
import os, sys
for d in (os.getcwd(), os.path.expanduser("~/iiswc-tutorial/notebooks/mb_lab")):
    if os.path.exists(os.path.join(d, "mb_lab.py")):
        sys.path.insert(0, d)
import mb_lab as lab
lab.use_runs(lab.solved_runs())     # read the stored runs, not this seat's
LLM_RUN, MY_RUN = "20260925-201916-maxpool2d_s8", "20260925-202434-maxpool2d_s8"

## 1 · MBP.MAX8, by hand

A 2×2 max pool outputs the largest of four neighbouring bytes. The plain C kernel does that one byte at a
time: **118 cycles per output** on the board. MBP.MAX8 compares **eight pairs of bytes in one instruction**.
Here is the trick the fast kernel uses, on random bytes: two rows in, four outputs out, two instructions.

In [ ]:
lab.accelerator(7)

**✏️ Your turn.** `lab.max8(a, b)` is MBP.MAX8 in Python: eight lanes, the maximum of each pair. Pick two rows
of eight int8 values (from -128 to 127), **predict** the answer, then run the cell.

In [ ]:
row0 = [12, -7, 100, 3, -128, 55, 0, 9]
row1 = [-4, 20, 99, 3, -1, -55, 127, 8]
my_prediction = [12, 20, 100, 3, -1, 55, 127, 9]
print("MBP.MAX8 says:", lab.max8(row0, row1), "  you said:", my_prediction, "  ",
      "✓" if lab.max8(row0, row1) == my_prediction else "✗")

## 2 · The LLM rewrites the kernel, with your FPGA in the loop (5–8 minutes)

Each round, the LLM writes a few kernels. **Spike**, a simulator, checks each one and times it in seconds.
Then the best one of the round is built for your board and **runs on your FPGA**, and the LLM is told the
real cycle count for its next round. Blue bars are spike, orange bars are your FPGA, and the dashed outline
is the same kernel with the accelerator switched off.

**✏️ First, guess:** how many times faster will the LLM's kernel be on your FPGA?

In [ ]:
my_guess = 10

In [ ]:
lab.show(LLM_RUN)   # what lab.go("maxpool2d_s8") drew when this run ended

## 3 · Where the speedup came from

The verdict shows two arms of the same kernel: MBP **off**, where `mb_pext_max8` becomes its C model
but the packed 8-byte loads remain, and MBP **on**. Read the first as *the packed dataflow with the SIMD
emulated*, not as *the loop on its own* — the loop on its own is the same kernel with `use_mbp` forced to
0, and measured on the bench board that arm is **0.97×**, three percent *slower* than the reference. So
the win here is not two independent factors multiplied together; almost all of it is the instruction.

Spike's number is higher than the board's. Spike charges one cycle per instruction and knows nothing
about memory, and once MAX8 makes the compute eight times denser, memory is what's left.

In [ ]:
lab.kernels(LLM_RUN)

In [ ]:
lab.calls(LLM_RUN)

### The tools, and every command the run executed

The lab is the usual tools, run in order:
- **ModelBlaster** turns the PyTorch model into an int8 graph and generates C kernels. With `--backend llm`
  it asks the LLM.
- **Zephyr's `west`** builds the images.
- **spike** simulates them.
- The **board's agent** runs them on the FPGA.

Here is every command the run executed, exactly. Paste any of them into a terminal.

In [ ]:
lab.commands(LLM_RUN)

## 4 · Write the kernel yourself (≈2 minutes per try)

`lab.start()` puts the unoptimized kernel in **`your-kernel/maxpool2d_s8.c`** (left, in the file browser). Its header has
the rules and four hints: read one at a time. Edit, save with **Ctrl-S**, and run `lab.try_kernel()`. It checks
your kernel on spike (a wrong one never reaches the board), then runs it on your FPGA three ways.
**Goal: `ON THE ACCELERATOR` and under 10 cycles per output.**

In [ ]:
lab.show(MY_RUN)    # lab.try_kernel() with the solution file
lab.scoreboard()

In [ ]:
lab.solution()

## 5 · More to try

* `lab.go("maxpool2d_s8", "--guide", "modelblaster")`: the LLM is **not** told about the accelerator. Does it find it?
* `lab.go("linear_s8")`: an int8 matrix multiply on MBP.DOT8, eight multiply-adds per instruction.
* `lab.go("gelu_s8")`: 43–49× faster with **no** accelerator at all. What did the LLM do instead?
* `lab.runs()` lists every run on this seat. `lab.verdict("<run>")`, `lab.kernels("<run>")`,
  `lab.calls("<run>")` and `lab.commands("<run>")` open one again.
* The same lab from a terminal (File → New → Terminal): `mb doctor`, `mb`, `mb try maxpool2d_s8`.